In [1]:
# STEP 1 — Weakly-supervised cough localisation on spectrogram images
# - Builds pseudo-label boxes (time axis) via energy + spectral-flux gating
# - Writes COCO JSON for each split (train_healthy, test_kaggle_healthy, test_kaggle_cancer)
# - Saves cropped cough patches (224x224) for downstream models
# - Prints per-source stats

import os, json, math, shutil
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from tqdm import tqdm

# ---------------- paths (edit if needed) ----------------
DIR_COUGHVID = Path("/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Processed_Regenerated/Cough_Audio_COUGHVID")
DIR_COSWARA  = Path("/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Processed_Regenerated/Cough_Audio_Coswera")
DIR_KAGGLE_H = Path("/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Processed_Regenerated/harmonized_additions/kaggle")
DIR_KAGGLE_C = Path("/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/Project_PreProess_harmonize/Project/Processed_Regenerated/harmonized_additions/cancer")

OUT_ROOT = Path("/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/CoughCrops")
OUT_COCO = OUT_ROOT / "coco"
OUT_CROPS = OUT_ROOT / "crops"  # subdirs created per split
for p in [OUT_COCO, OUT_CROPS]:
    p.mkdir(parents=True, exist_ok=True)

# ---------------- gather images ----------------
def list_images(root):
    exts = {".png",".jpg",".jpeg",".bmp",".tif",".tiff",".webp"}
    if not root.exists(): return []
    return [p for p in root.rglob("*") if p.suffix.lower() in exts]

imgs_coughvid = list_images(DIR_COUGHVID)
imgs_coswara  = list_images(DIR_COSWARA)
imgs_kaggle_h = list_images(DIR_KAGGLE_H)
imgs_kaggle_c = list_images(DIR_KAGGLE_C)

print("Found images:")
print(f"  COUGHVID healthy: {len(imgs_coughvid)}")
print(f"  Coswara  healthy: {len(imgs_coswara)}")
print(f"  Kaggle   healthy: {len(imgs_kaggle_h)}")
print(f"  Kaggle   cancer : {len(imgs_kaggle_c)}")

# ---------------- pseudo-boxer on spectrogram images ----------------
def smooth_1d(x, k=7):
    k = max(1, int(k))
    if k == 1: return x
    pad = k//2
    xp = np.pad(x, (pad,pad), mode="edge")
    w = np.ones(k, dtype=np.float32)/k
    return np.convolve(xp, w, mode="valid")

def find_segments_from_spectrogram_img(img_gray,
                                       energy_smooth=9,
                                       flux_smooth=9,
                                       thr_k=2.0,
                                       min_width_frac=0.01,
                                       merge_gap_frac=0.01,
                                       pad_frac=0.01):
    """
    img_gray: HxW float in [0,1]
    returns list of (x0, x1) on time axis (columns), inclusive-exclusive.
    """
    H, W = img_gray.shape
    # column energy (robust): mean over frequency
    e = img_gray.mean(axis=0)
    e = smooth_1d(e, energy_smooth)
    e = (e - np.median(e)) / (np.median(np.abs(e - np.median(e))) + 1e-8)  # robust z

    # spectral flux proxy from image: positive diff along time of mean energy
    d = np.diff(img_gray.mean(axis=0), prepend=0.0)
    d = np.maximum(d, 0.0)
    d = smooth_1d(d, flux_smooth)
    d = (d - np.median(d)) / (np.median(np.abs(d - np.median(d))) + 1e-8)

    s = 0.6*e + 0.4*d  # combine
    # adaptive threshold
    thr = np.median(s) + thr_k * (np.median(np.abs(s - np.median(s))) + 1e-8)

    mask = s > thr
    # group contiguous
    segs = []
    i = 0
    while i < W:
        if mask[i]:
            j = i+1
            while j < W and mask[j]: j += 1
            segs.append([i, j])
            i = j
        else:
            i += 1

    # merge close gaps
    min_gap = int(merge_gap_frac * W)
    merged = []
    for seg in segs:
        if not merged:
            merged.append(seg)
        else:
            if seg[0] - merged[-1][1] <= min_gap:
                merged[-1][1] = seg[1]
            else:
                merged.append(seg)

    # expand/pad and drop too-narrow
    pad = int(pad_frac * W)
    minw = max(2, int(min_width_frac * W))
    final = []
    for x0, x1 in merged:
        x0p = max(0, x0 - pad)
        x1p = min(W, x1 + pad)
        if x1p - x0p >= minw:
            final.append((x0p, x1p))
    return final

def load_gray_float(path):
    im = Image.open(path).convert("L")
    arr = np.asarray(im, dtype=np.float32)
    if arr.max() > 1.0:
        arr = arr / 255.0
    return arr

def save_square_224(crop_arr, out_path):
    # crop_arr: H x W in [0,1]
    H, W = crop_arr.shape
    # make square by padding horizontally or vertically
    if H == W:
        sq = crop_arr
    elif H > W:
        pad = (H - W) // 2
        sq = np.pad(crop_arr, ((0,0),(pad, H - W - pad)), mode="edge")
    else:
        pad = (W - H) // 2
        sq = np.pad(crop_arr, ((pad, W - H - pad),(0,0)), mode="edge")
    im = Image.fromarray((np.clip(sq,0,1)*255).astype(np.uint8))
    im = im.resize((224,224), Image.BILINEAR)
    im.save(out_path)

# ---------------- build splits & process ----------------
SPLITS = {
    "train_healthy": imgs_coughvid + imgs_coswara,           # healthy training pool
    "test_healthy":  imgs_kaggle_h,                           # held-out healthy
    "test_cancer":   imgs_kaggle_c                            # held-out cancer
}

all_stats = {}
all_ann = {}
for split, img_list in SPLITS.items():
    images, annotations = [], []
    img_id = 0
    ann_id = 0
    n_boxes = 0
    n_imgs = 0
    per_img_boxes = []

    out_dir = OUT_CROPS / split
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    for p in tqdm(img_list, desc=f"Processing {split}"):
        try:
            arr = load_gray_float(p)
            H, W = arr.shape
            segs = find_segments_from_spectrogram_img(arr)
            # record COCO-like
            images.append({"id": img_id, "file_name": str(p), "height": int(H), "width": int(W)})
            for (x0,x1) in segs:
                w = int(x1 - x0); h = int(H)
                bbox = [int(x0), 0, w, h]  # x,y,w,h ; y spans full mel axis
                annotations.append({
                    "id": ann_id,
                    "image_id": img_id,
                    "category_id": 0,
                    "bbox": bbox,
                    "area": int(w*h),
                    "iscrowd": 0
                })
                ann_id += 1
                # save crop as 224x224
                crop = arr[:, x0:x1]
                rel = Path(p).with_suffix(".png").name
                out_path = out_dir / f"{Path(rel).stem}__x{x0}-{x1}.png"
                save_square_224(crop, out_path)
                n_boxes += 1
            per_img_boxes.append(len(segs))
            img_id += 1
            n_imgs += 1
        except Exception as e:
            print(f"[warn] failed {p}: {e}")

    coco = {
        "images": images,
        "annotations": annotations,
        "categories": [{"id":0,"name":"cough"}]
    }
    with open(OUT_COCO / f"{split}.json", "w") as f:
        json.dump(coco, f)

    all_ann[split] = coco
    all_stats[split] = {
        "n_images": n_imgs,
        "n_boxes": n_boxes,
        "boxes_per_image_mean": float(np.mean(per_img_boxes)) if per_img_boxes else 0.0,
        "boxes_per_image_median": float(np.median(per_img_boxes)) if per_img_boxes else 0.0
    }

# ---------------- summary & paths ----------------
print("\n=== Pseudo-cough localisation summary ===")
for split, st in all_stats.items():
    print(f"{split:>14}: images={st['n_images']:4d} | boxes={st['n_boxes']:4d} "
          f"| boxes/img mean={st['boxes_per_image_mean']:.2f} med={st['boxes_per_image_median']:.2f}")

print("\n✔️ Crops saved under:", OUT_CROPS.resolve())
print("   -", (OUT_CROPS / "train_healthy").resolve())
print("   -", (OUT_CROPS / "test_healthy").resolve())
print("   -", (OUT_CROPS / "test_cancer").resolve())
print("✔️ COCO JSON saved under:", (OUT_COCO).resolve())
print("   -", (OUT_COCO / "train_healthy.json").resolve())
print("   -", (OUT_COCO / "test_healthy.json").resolve())
print("   -", (OUT_COCO / "test_cancer.json").resolve())


Found images:
  COUGHVID healthy: 837
  Coswara  healthy: 2809
  Kaggle   healthy: 24
  Kaggle   cancer : 30


Processing test_cancer: 100%|██████████| 30/30 [00:00<00:00, 32.66it/s]


=== Pseudo-cough localisation summary ===
 train_healthy: images=3646 | boxes=17959 | boxes/img mean=4.93 med=5.00
  test_healthy: images=  24 | boxes=  93 | boxes/img mean=3.88 med=4.00
   test_cancer: images=  30 | boxes= 164 | boxes/img mean=5.47 med=5.00

✔️ Crops saved under: /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/CoughCrops/crops
   - /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/CoughCrops/crops/train_healthy
   - /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/CoughCrops/crops/test_healthy
   - /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/CoughCrops/crops/test_cancer
✔️ COCO JSON saved under: /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/CoughCrops/coco
   - /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/CoughCrops/coco/train_healthy.json
   - /workspaces/cmp9137-advanced

In [1]:
# STEP 2 — LOSO anomaly on CROPPED coughs (train healthy; test Kaggle healthy vs cancer)
# Method: ResNet-18 frozen embeddings -> Gaussian (Ledoit-Wolf) Mahalanobis anomaly
# Aggregation: per recording = max crop score
# Outputs: ROC-AUC, PR-AUC (cancer+), F1 @ 5% FPR (per-record), confusion matrix

import os, math
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch, torchvision as tv
from torch.utils.data import Dataset, DataLoader

from sklearn.covariance import LedoitWolf
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, confusion_matrix, precision_recall_curve

# ------------ paths (must match Step 1 outputs) ------------
ROOT = Path("/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/CoughCrops/crops")
DIR_TR = ROOT / "train_healthy"
DIR_TH = ROOT / "test_healthy"
DIR_TC = ROOT / "test_cancer"

assert DIR_TR.exists() and DIR_TH.exists() and DIR_TC.exists(), "Crop folders not found. Run Step 1 first."

# ------------ utilities ------------
def list_imgs(d):
    exts = {".png",".jpg",".jpeg",".bmp",".tif",".tiff",".webp"}
    return [p for p in d.rglob("*") if p.suffix.lower() in exts]

def group_id_from_name(p: Path):
    # crops are saved as "<orig-stem>__xSTART-END.png"
    s = p.stem
    if "__x" in s:
        return s.split("__x")[0]
    return s

class ImgDataset(Dataset):
    def __init__(self, paths):
        self.paths = paths
        self.tf = tv.transforms.Compose([
            tv.transforms.Grayscale(num_output_channels=3),
            tv.transforms.ToTensor(),
            tv.transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        x = Image.open(self.paths[i]).convert("L")
        x = self.tf(x)
        return x, i  # return index to align features to paths

@torch.no_grad()
def embed(paths, backbone, batch=128, num_workers=0, device="cpu"):
    ds = ImgDataset(paths)
    dl = DataLoader(ds, batch_size=batch, shuffle=False, num_workers=num_workers)
    feats = np.zeros((len(paths), 512), dtype=np.float32)
    idx0 = 0
    for xb, ii in tqdm(dl, desc="Embedding", leave=False):
        xb = xb.to(device)
        f = backbone(xb).flatten(1).cpu().numpy().astype(np.float32)
        feats[idx0:idx0+f.shape[0]] = f
        idx0 += f.shape[0]
    return feats

def fit_gaussian_mahalanobis(X):
    mu = X.mean(axis=0, keepdims=True)
    Xc = X - mu
    lw = LedoitWolf().fit(Xc)
    prec = lw.precision_
    return mu, prec

def mahalanobis_scores(X, mu, prec):
    Xc = X - mu
    # score = (x - mu)^T * prec * (x - mu)
    return np.einsum("ij,jk,ik->i", Xc, prec, Xc)

def aggregate_per_record(paths, scores):
    groups = defaultdict(list)
    for p,s in zip(paths, scores):
        gid = group_id_from_name(p)
        groups[gid].append(float(s))
    # aggregate by max (most abnormal burst)
    gids = []
    agg_scores = []
    for gid, vals in groups.items():
        gids.append(gid)
        agg_scores.append(max(vals))
    return np.array(gids), np.array(agg_scores)

def f1_at_fpr(y_true, y_score, target_fpr=0.05):
    # Choose threshold as (1 - target_fpr) quantile of scores among negatives
    neg_scores = y_score[y_true==0]
    if len(neg_scores)==0:
        return None
    thr = np.quantile(neg_scores, 1 - target_fpr)  # 95th percentile for 5% FPR target
    y_pred = (y_score >= thr).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0,1])
    f1 = f1_score(y_true, y_pred)
    return thr, f1, cm

# ------------ collect file lists ------------
tr_paths = list_imgs(DIR_TR)
th_paths = list_imgs(DIR_TH)
tc_paths = list_imgs(DIR_TC)

print("--- Step 1: Partitioning Data (cropped) ---")
print(f"Train (healthy crops): {len(tr_paths)}")
print(f"Test healthy (Kaggle crops): {len(th_paths)}")
print(f"Test cancer  (Kaggle crops): {len(tc_paths)}")

# ------------ feature extractor (frozen ResNet-18) ------------
device = "cuda" if torch.cuda.is_available() else "cpu"
resnet = tv.models.resnet18(weights=tv.models.ResNet18_Weights.IMAGENET1K_V1).to(device).eval()
backbone = torch.nn.Sequential(*(list(resnet.children())[:-1]))  # -> (N,512,1,1)

# To keep training tractable if train crops are huge, subsample up to MAX_TRAIN_CROPS uniformly by recording
MAX_TRAIN_CROPS = 12000
if len(tr_paths) > MAX_TRAIN_CROPS:
    # group by recording, take up to k per group
    rec2crops = defaultdict(list)
    for p in tr_paths:
        rec2crops[group_id_from_name(p)].append(p)
    # compute per-rec cap
    k = max(3, int(MAX_TRAIN_CROPS / max(1, len(rec2crops))))
    subsampled = []
    rng = np.random.default_rng(123)
    for rec, lst in rec2crops.items():
        if len(lst) <= k:
            subsampled += lst
        else:
            idx = rng.choice(len(lst), size=k, replace=False)
            subsampled += [lst[i] for i in idx]
    tr_paths_use = subsampled
    print(f"Subsampled train crops -> {len(tr_paths_use)} (≈{k}/recording)")
else:
    tr_paths_use = tr_paths

# ------------ embed ------------
print("\n--- Step 2: Extracting features ---")
X_tr = embed(tr_paths_use, backbone, batch=128, num_workers=0, device=device)
X_th = embed(th_paths, backbone, batch=128, num_workers=0, device=device)
X_tc = embed(tc_paths, backbone, batch=128, num_workers=0, device=device)

# ------------ fit anomaly model on healthy crops ------------
print("\n--- Step 3: Fitting Gaussian (Ledoit–Wolf) on healthy crops ---")
mu, prec = fit_gaussian_mahalanobis(X_tr)

# ------------ score test crops ------------
s_th = mahalanobis_scores(X_th, mu, prec)
s_tc = mahalanobis_scores(X_tc, mu, prec)

# ------------ aggregate to recording level ------------
print("\n--- Step 4: Aggregating per recording (max over crops) ---")
gid_th, rec_th = aggregate_per_record(th_paths, s_th)
gid_tc, rec_tc = aggregate_per_record(tc_paths, s_tc)

# build per-record arrays
y_true = np.concatenate([np.zeros_like(rec_th, dtype=int), np.ones_like(rec_tc, dtype=int)])
y_score = np.concatenate([rec_th, rec_tc])

# ------------ metrics ------------
print("\n--- FINAL RESULTS (per recording) ---")
roc = roc_auc_score(y_true, y_score)
pr  = average_precision_score(y_true, y_score)  # cancer positive
thr_5fpr, f1_5fpr, cm_5fpr = f1_at_fpr(y_true, y_score, target_fpr=0.05)
# also report max-F1 operating point for context
ps, rs, ts = precision_recall_curve(y_true, y_score)
f1s = (2*ps*rs)/(ps+rs+1e-12)
i = np.nanargmax(f1s)
tau_star = float(ts[max(0, i-1)]) if i>0 else 0.0
yhat_star = (y_score >= tau_star).astype(int)
cm_star = confusion_matrix(y_true, yhat_star, labels=[0,1])
f1_star = f1_score(y_true, yhat_star)

print(f"ROC-AUC: {roc:.3f}")
print(f"PR-AUC (Cancer is Positive Class): {pr:.3f}")
print(f"F1-Score @ 5% FPR: {f1_5fpr:.3f} (threshold={thr_5fpr:.4f})")
print("Confusion matrix @ 5% FPR (rows=true [healthy,cancer], cols=pred):")
print(cm_5fpr)
print(f"\nF1-Score @ τ* (max-F1): {f1_star:.3f} (threshold={tau_star:.4f})")
print("Confusion matrix @ τ* (rows=true [healthy,cancer], cols=pred):")
print(cm_star)


--- Step 1: Partitioning Data (cropped) ---
Train (healthy crops): 17959
Test healthy (Kaggle crops): 93
Test cancer  (Kaggle crops): 164
Subsampled train crops -> 10434 (≈3/recording)

--- Step 2: Extracting features ---



--- Step 3: Fitting Gaussian (Ledoit–Wolf) on healthy crops ---

--- Step 4: Aggregating per recording (max over crops) ---

--- FINAL RESULTS (per recording) ---
ROC-AUC: 0.303
PR-AUC (Cancer is Positive Class): 0.438
F1-Score @ 5% FPR: 0.000 (threshold=6573.2219)
Confusion matrix @ 5% FPR (rows=true [healthy,cancer], cols=pred):
[[22  2]
 [30  0]]

F1-Score @ τ* (max-F1): 0.723 (threshold=1566.9944)
Confusion matrix @ τ* (rows=true [healthy,cancer], cols=pred):
[[ 1 23]
 [ 0 30]]


In [2]:
# STEP 2a — Aggregation & inversion audit for LOSO anomaly on cropped coughs
# Evaluates: aggregators ∈ {max, mean, p95, top2_mean, top3_mean} × {original, inverted scores}
# Metrics: ROC-AUC, PR-AUC (cancer+), F1@5%FPR, F1@τ* (max-F1), with confusion matrices suppressed for brevity

import numpy as np, pandas as pd, math, os
from pathlib import Path
from collections import defaultdict
from PIL import Image
import torch, torchvision as tv
from torch.utils.data import Dataset, DataLoader
from sklearn.covariance import LedoitWolf
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, confusion_matrix, precision_recall_curve

# ---------- config & paths ----------
ROOT = Path("/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/CoughCrops/crops")
DIR_TR = ROOT / "train_healthy"
DIR_TH = ROOT / "test_healthy"
DIR_TC = ROOT / "test_cancer"
device = "cuda" if torch.cuda.is_available() else "cpu"

# ---------- helpers ----------
def list_imgs(d):
    exts = {".png",".jpg",".jpeg",".bmp",".tif",".tiff",".webp"}
    return [p for p in d.rglob("*") if p.suffix.lower() in exts]

def group_id_from_name(p: Path):
    s = p.stem
    return s.split("__x")[0] if "__x" in s else s

class ImgDataset(Dataset):
    def __init__(self, paths):
        self.paths = paths
        self.tf = tv.transforms.Compose([
            tv.transforms.Grayscale(num_output_channels=3),
            tv.transforms.ToTensor(),
            tv.transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        x = Image.open(self.paths[i]).convert("L")
        return self.tf(x), i

@torch.no_grad()
def embed(paths, backbone, batch=128, num_workers=0, device="cpu"):
    ds = ImgDataset(paths)
    dl = DataLoader(ds, batch_size=batch, shuffle=False, num_workers=num_workers)
    feats = np.zeros((len(paths), 512), dtype=np.float32)
    idx0 = 0
    for xb, ii in dl:
        xb = xb.to(device)
        f = backbone(xb).flatten(1).cpu().numpy().astype(np.float32)
        feats[idx0:idx0+f.shape[0]] = f
        idx0 += f.shape[0]
    return feats

def fit_gaussian_mahalanobis(X):
    mu = X.mean(axis=0, keepdims=True)
    Xc = X - mu
    lw = LedoitWolf().fit(Xc)
    return mu, lw.precision_

def mahalanobis_scores(X, mu, prec):
    Xc = X - mu
    return np.einsum("ij,jk,ik->i", Xc, prec, Xc)

def aggregate_record_scores(paths, scores, agg="max"):
    groups = defaultdict(list)
    for p,s in zip(paths, scores):
        groups[group_id_from_name(p)].append(float(s))
    gids, agg_scores = [], []
    for gid, vals in groups.items():
        v = sorted(vals, reverse=True)
        if   agg == "max":      a = v[0]
        elif agg == "mean":     a = float(np.mean(v))
        elif agg == "p95":      a = float(np.percentile(v, 95))
        elif agg == "top2_mean":a = float(np.mean(v[:2])) if len(v)>=2 else float(np.mean(v))
        elif agg == "top3_mean":a = float(np.mean(v[:3])) if len(v)>=3 else float(np.mean(v))
        else: raise ValueError("unknown agg")
        gids.append(gid); agg_scores.append(a)
    return np.array(gids), np.array(agg_scores)

def f1_at_fpr(y_true, y_score, target_fpr=0.05):
    neg = y_score[y_true==0]
    if len(neg)==0:
        return np.nan, np.nan, np.array([[0,0],[0,0]])
    thr = np.quantile(neg, 1 - target_fpr)
    yhat = (y_score >= thr).astype(int)
    return float(thr), f1_score(y_true, yhat), confusion_matrix(y_true, yhat, labels=[0,1])

def f1_at_tau_star(y_true, y_score):
    ps, rs, ts = precision_recall_curve(y_true, y_score)
    f1s = (2*ps*rs)/(ps+rs+1e-12)
    i = np.nanargmax(f1s)
    tau = float(ts[max(0, i-1)]) if i>0 else 0.0
    yhat = (y_score >= tau).astype(int)
    return float(tau), f1_score(y_true, yhat), confusion_matrix(y_true, yhat, labels=[0,1])

# ---------- try to reuse previous variables; else recompute ----------
need_recompute = not all(k in globals() for k in ["mu","prec","th_paths","tc_paths","s_th","s_tc"])
if need_recompute:
    # list files
    tr_paths = list_imgs(DIR_TR)
    th_paths = list_imgs(DIR_TH)
    tc_paths = list_imgs(DIR_TC)
    # backbone
    resnet = tv.models.resnet18(weights=tv.models.ResNet18_Weights.IMAGENET1K_V1).to(device).eval()
    backbone = torch.nn.Sequential(*(list(resnet.children())[:-1]))
    # (optional) subsample train crops if huge
    MAX_TRAIN_CROPS = 12000
    if len(tr_paths) > MAX_TRAIN_CROPS:
        from collections import defaultdict
        rec2 = defaultdict(list)
        for p in tr_paths: rec2[group_id_from_name(p)].append(p)
        k = max(3, int(MAX_TRAIN_CROPS / max(1,len(rec2))))
        rng = np.random.default_rng(123)
        subsampled = []
        for rec,lst in rec2.items():
            if len(lst) <= k: subsampled += lst
            else: subsampled += [lst[i] for i in rng.choice(len(lst), size=k, replace=False)]
        tr_paths_use = subsampled
    else:
        tr_paths_use = tr_paths
    # embed
    X_tr = embed(tr_paths_use, backbone, device=device)
    X_th = embed(th_paths, backbone, device=device)
    X_tc = embed(tc_paths, backbone, device=device)
    # fit gaussian & score
    mu, prec = fit_gaussian_mahalanobis(X_tr)
    s_th = mahalanobis_scores(X_th, mu, prec)
    s_tc = mahalanobis_scores(X_tc, mu, prec)
    # export names for downstream
    globals().update({"tr_paths":tr_paths_use, "th_paths":th_paths, "tc_paths":tc_paths})

# ---------- evaluate aggregators & inversion ----------
AGGS = ["max","mean","p95","top2_mean","top3_mean"]
rows = []

for agg in AGGS:
    # aggregate per recording
    gid_th, rec_th = aggregate_record_scores(th_paths, s_th, agg=agg)
    gid_tc, rec_tc = aggregate_record_scores(tc_paths, s_tc, agg=agg)

    for invert in [False, True]:
        y_true = np.concatenate([np.zeros_like(rec_th, dtype=int), np.ones_like(rec_tc, dtype=int)])
        y_score = np.concatenate([rec_th, rec_tc])
        if invert:
            y_score = -y_score

        try:
            roc = roc_auc_score(y_true, y_score)
        except Exception:
            roc = np.nan
        try:
            pr = average_precision_score(y_true, y_score)
        except Exception:
            pr = np.nan

        thr5, f1_5, cm5 = f1_at_fpr(y_true, y_score, target_fpr=0.05)
        tau, f1_star, cm_star = f1_at_tau_star(y_true, y_score)

        rows.append({
            "aggregator": agg,
            "inverted": invert,
            "ROC_AUC": roc,
            "PR_AUC": pr,
            "F1_at_5pctFPR": f1_5,
            "F1_at_tau_star": f1_star,
            "thr@5%FPR": thr5,
            "tau_star": tau,
            "CM@5%FPR": cm5.tolist(),
            "CM@tau*": cm_star.tolist(),
        })

df = pd.DataFrame(rows)
df_sorted = df.sort_values(["ROC_AUC","F1_at_5pctFPR","F1_at_tau_star"], ascending=False, na_position="last")
pd.set_option("display.max_colwidth", 120)
print("\n=== Aggregation × Inversion Audit (per recording) ===")
print(df_sorted.round(3).to_string(index=False))



=== Aggregation × Inversion Audit (per recording) ===
aggregator  inverted  ROC_AUC  PR_AUC  F1_at_5pctFPR  F1_at_tau_star  thr@5%FPR  tau_star           CM@5%FPR             CM@tau*
      mean      True    0.756   0.673            0.0           0.857  -1279.530 -2121.959 [[22, 2], [30, 0]] [[14, 10], [0, 30]]
 top3_mean      True    0.708   0.651            0.0           0.800  -1369.027 -2717.479 [[22, 2], [30, 0]]  [[9, 15], [0, 30]]
 top2_mean      True    0.703   0.649            0.0           0.800  -1493.892 -2892.610 [[21, 3], [30, 0]]  [[9, 15], [0, 30]]
       p95      True    0.703   0.651            0.0           0.811  -1545.064 -3081.792 [[21, 3], [30, 0]] [[10, 14], [0, 30]]
       max      True    0.697   0.649            0.0           0.811  -1566.994 -3098.609 [[21, 3], [30, 0]] [[10, 14], [0, 30]]
       max     False    0.303   0.438            0.0           0.723   6573.222  1566.994 [[22, 2], [30, 0]]  [[1, 23], [0, 30]]
       p95     False    0.297   0.435     

In [3]:
# EXPORTS — LOSO anomaly on cropped coughs (mean aggregation + inverted scores)
# Saves: ROC/PR, CMs (tau*, 10% FPR, 20% FPR), metrics.csv, per_record_scores.csv, CIs, config.json

import os, json, math
from pathlib import Path
from collections import defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch, torchvision as tv
from torch.utils.data import Dataset, DataLoader
from sklearn.covariance import LedoitWolf
from sklearn.metrics import (
    roc_curve, auc, precision_recall_curve, confusion_matrix, classification_report,
    roc_auc_score, average_precision_score, f1_score
)

# ---------------- paths ----------------
ROOT_CROPS = Path("/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/CoughCrops/crops")
DIR_TR = ROOT_CROPS / "train_healthy"
DIR_TH = ROOT_CROPS / "test_healthy"
DIR_TC = ROOT_CROPS / "test_cancer"
OUT = Path("/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/CoughCrops/exports_loso_cropped_meaninv")
OUT.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"

# ---------------- helpers ----------------
def list_imgs(d):
    exts = {".png",".jpg",".jpeg",".bmp",".tif",".tiff",".webp"}
    return [p for p in d.rglob("*") if p.suffix.lower() in exts]

def group_id_from_name(p: Path):
    s = p.stem
    return s.split("__x")[0] if "__x" in s else s

class ImgDataset(Dataset):
    def __init__(self, paths):
        self.paths = paths
        self.tf = tv.transforms.Compose([
            tv.transforms.Grayscale(num_output_channels=3),
            tv.transforms.ToTensor(),
            tv.transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        x = Image.open(self.paths[i]).convert("L")
        return self.tf(x), i

@torch.no_grad()
def embed(paths, backbone, batch=128, num_workers=0, device="cpu"):
    ds = ImgDataset(paths)
    dl = DataLoader(ds, batch_size=batch, shuffle=False, num_workers=num_workers)
    feats = np.zeros((len(paths), 512), dtype=np.float32)
    idx0 = 0
    for xb, ii in dl:
        xb = xb.to(device)
        f = backbone(xb).flatten(1).cpu().numpy().astype(np.float32)
        feats[idx0:idx0+f.shape[0]] = f
        idx0 += f.shape[0]
    return feats

def fit_gaussian_mahalanobis(X):
    mu = X.mean(axis=0, keepdims=True)
    Xc = X - mu
    lw = LedoitWolf().fit(Xc)
    return mu, lw.precision_

def mahalanobis_scores(X, mu, prec):
    Xc = X - mu
    return np.einsum("ij,jk,ik->i", Xc, prec, Xc)

def aggregate_mean(paths, scores):
    groups = defaultdict(list)
    for p,s in zip(paths, scores):
        groups[group_id_from_name(p)].append(float(s))
    gids, agg = [], []
    for gid, vals in groups.items():
        gids.append(gid); agg.append(float(np.mean(vals)))
    return np.array(gids), np.array(agg)

def pick_tau_maxF1(y, s):
    p, r, t = precision_recall_curve(y, s)
    f1 = 2*p*r/(p+r+1e-12)
    i = np.nanargmax(f1)
    tau = float(t[max(0, i-1)]) if i>0 else 0.0
    return tau

def threshold_at_fpr(y, s, target_fpr=0.10):
    # choose threshold using negatives (healthy): quantile 1-target_fpr
    neg = s[y==0]
    thr = np.quantile(neg, 1 - target_fpr)
    return float(thr)

def eval_at_tau(y, s, tau):
    yhat = (s >= tau).astype(int)
    cm  = confusion_matrix(y, yhat, labels=[0,1])
    rep = classification_report(y, yhat, target_names=['healthy','cancer'], digits=4, output_dict=True)
    return cm, rep, f1_score(y, yhat)

def save_cm(cm, name):
    plt.figure(figsize=(4,3.2), dpi=130)
    plt.imshow(cm, cmap="Blues")
    plt.title(name)
    plt.xticks([0,1], ["healthy","cancer"])
    plt.yticks([0,1], ["healthy","cancer"])
    for i in range(2):
        for j in range(2):
            plt.text(j,i,str(cm[i,j]), ha="center", va="center")
    plt.xlabel("Predicted"); plt.ylabel("True"); plt.tight_layout()
    plt.savefig(OUT/f"{name.replace(' ','_').replace('%','p')}.png"); plt.close()

# ---------------- reuse or recompute ----------------
recompute = not all(k in globals() for k in ["mu","prec","th_paths","tc_paths","s_th","s_tc"])
if recompute:
    tr_paths = list_imgs(DIR_TR)
    th_paths = list_imgs(DIR_TH)
    tc_paths = list_imgs(DIR_TC)
    # backbone
    resnet = tv.models.resnet18(weights=tv.models.ResNet18_Weights.IMAGENET1K_V1).to(device).eval()
    backbone = torch.nn.Sequential(*(list(resnet.children())[:-1]))
    # subsample train crops if huge
    MAX_TRAIN_CROPS = 12000
    if len(tr_paths) > MAX_TRAIN_CROPS:
        rec2 = defaultdict(list)
        for p in tr_paths: rec2[group_id_from_name(p)].append(p)
        k = max(3, int(MAX_TRAIN_CROPS / max(1,len(rec2))))
        rng = np.random.default_rng(123)
        sub = []
        for rec,lst in rec2.items():
            if len(lst) <= k: sub += lst
            else: sub += [lst[i] for i in rng.choice(len(lst), size=k, replace=False)]
        tr_paths_use = sub
    else:
        tr_paths_use = tr_paths
    # embed & fit
    X_tr = embed(tr_paths_use, backbone, device=device)
    mu, prec = fit_gaussian_mahalanobis(X_tr)
    # score tests
    X_th = embed(th_paths, backbone, device=device)
    X_tc = embed(tc_paths, backbone, device=device)
    s_th = mahalanobis_scores(X_th, mu, prec)
    s_tc = mahalanobis_scores(X_tc, mu, prec)
    # publish to globals to allow re-use later in session
    globals().update({"th_paths":th_paths, "tc_paths":tc_paths, "mu":mu, "prec":prec, "s_th":s_th, "s_tc":s_tc})

# ---------------- aggregate per recording (mean) + invert ----------------
gid_th, rec_th = aggregate_mean(th_paths, s_th)
gid_tc, rec_tc = aggregate_mean(tc_paths, s_tc)

y_true  = np.concatenate([np.zeros_like(rec_th, dtype=int), np.ones_like(rec_tc, dtype=int)])
y_score = -np.concatenate([rec_th, rec_tc])  # invert: higher = more likely cancer

# ---------------- curves ----------------
fpr, tpr, _ = roc_curve(y_true, y_score)
roc_auc = auc(fpr, tpr)
prec, rec, _ = precision_recall_curve(y_true, y_score)
pr_auc = auc(rec, prec)

plt.figure(figsize=(6,5), dpi=130)
plt.plot(fpr, tpr, label=f"ROC (AUC={roc_auc:.3f})")
plt.plot([0,1],[0,1],'--',alpha=0.4)
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate"); plt.title("LOSO (cropped) — ROC (mean+invert)")
plt.legend(); plt.tight_layout()
plt.savefig(OUT/"roc_curve.png"); plt.close()

plt.figure(figsize=(6,5), dpi=130)
plt.plot(rec, prec, label=f"PR (AUC={pr_auc:.3f})")
plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("LOSO (cropped) — Precision–Recall (mean+invert)")
plt.legend(); plt.tight_layout()
plt.savefig(OUT/"pr_curve.png"); plt.close()

# ---------------- operating points ----------------
tau_star = pick_tau_maxF1(y_true, y_score)
thr_fpr10 = threshold_at_fpr(y_true, y_score, target_fpr=0.10)
thr_fpr20 = threshold_at_fpr(y_true, y_score, target_fpr=0.20)

cm_star, rep_star, f1_star = eval_at_tau(y_true, y_score, tau_star)
cm_fpr10, rep_fpr10, f1_fpr10 = eval_at_tau(y_true, y_score, thr_fpr10)
cm_fpr20, rep_fpr20, f1_fpr20 = eval_at_tau(y_true, y_score, thr_fpr20)

save_cm(cm_star,  "cm_tau_star_maxF1")
save_cm(cm_fpr10, "cm_tau_10%_FPR")
save_cm(cm_fpr20, "cm_tau_20%_FPR")

# ---------------- metrics table ----------------
def row(name, tau, cm, rep, f1):
    TN, FP, FN, TP = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
    return {
        "operating_point": name,
        "threshold": float(tau),
        "ROC_AUC": float(roc_auc),
        "PR_AUC": float(pr_auc),
        "F1": float(f1),
        "precision_pos": float(rep["cancer"]["precision"]),
        "recall_pos": float(rep["cancer"]["recall"]),
        "specificity": float(TN / max(1, (TN+FP))),
        "TP": int(TP), "FN": int(FN), "FP": int(FP), "TN": int(TN)
    }

rows = [
    row("tau_star_maxF1", tau_star,  cm_star,  rep_star,  f1_star),
    row("tau_10%_FPR",   thr_fpr10, cm_fpr10, rep_fpr10, f1_fpr10),
    row("tau_20%_FPR",   thr_fpr20, cm_fpr20, rep_fpr20, f1_fpr20),
]
df_metrics = pd.DataFrame(rows)
df_metrics.to_csv(OUT/"metrics_loso_cropped_meaninv.csv", index=False)

# ---------------- per-record CSV ----------------
ids_h, ids_c = gid_th.tolist(), gid_tc.tolist()
scores_h = y_score[:len(ids_h)]
scores_c = y_score[len(ids_h):]
df_rec = pd.DataFrame({
    "recording_id": ids_h + ids_c,
    "true": [0]*len(ids_h) + [1]*len(ids_c),
    "score_mean_inverted": list(scores_h) + list(scores_c)
})
df_rec.to_csv(OUT/"per_record_scores_meaninv.csv", index=False)

# ---------------- bootstrap CIs ----------------
rng = np.random.RandomState(123)
N_BOOT = 2000
auc_boot, ap_boot = [], []
n = len(y_true)
for _ in range(N_BOOT):
    idx = rng.randint(0, n, n)
    if len(np.unique(y_true[idx])) < 2:
        continue
    auc_boot.append(roc_auc_score(y_true[idx], y_score[idx]))
    ap_boot.append(average_precision_score(y_true[idx], y_score[idx]))

def ci(x):
    x = np.array(x); 
    return float(np.percentile(x, 2.5)), float(np.percentile(x, 97.5))

auc_ci = ci(auc_boot) if auc_boot else (np.nan, np.nan)
ap_ci  = ci(ap_boot)  if ap_boot  else (np.nan, np.nan)

with open(OUT/"auc_ap_bootstrap_CI.txt","w") as f:
    f.write(f"AUC={roc_auc:.4f}  95% CI≈[{auc_ci[0]:.4f}, {auc_ci[1]:.4f}]\n")
    f.write(f"AP ={pr_auc:.4f}  95% CI≈[{ap_ci[0]:.4f}, {ap_ci[1]:.4f}]\n")

# ---------------- config snapshot ----------------
config = {
    "date": datetime.utcnow().isoformat()+"Z",
    "experiment": "LOSO anomaly on cropped coughs",
    "aggregation": "mean",
    "inverted_scores": True,
    "feature_extractor": "ResNet-18 (ImageNet), frozen",
    "anomaly_model": "Gaussian (Ledoit-Wolf) Mahalanobis on healthy crops",
    "train_crops_dir": str(DIR_TR),
    "test_healthy_crops_dir": str(DIR_TH),
    "test_cancer_crops_dir": str(DIR_TC),
    "n_train_crops_used": int(len(globals().get("tr_paths", list_imgs(DIR_TR)))),
    "n_test_healthy_crops": int(len(list_imgs(DIR_TH))),
    "n_test_cancer_crops": int(len(list_imgs(DIR_TC))),
    "per_record": {"healthy": int(len(set(map(group_id_from_name, list_imgs(DIR_TH))))),
                   "cancer":  int(len(set(map(group_id_from_name, list_imgs(DIR_TC)))))}
}
with open(OUT/"config_meaninv.json","w") as f:
    json.dump(config, f, indent=2)

# ---------------- summary ----------------
print("Saved to:", OUT.resolve())
print(" - roc_curve.png, pr_curve.png")
print(" - cm_tau_star_maxF1.png, cm_tau_10%_FPR.png, cm_tau_20%_FPR.png")
print(" - metrics_loso_cropped_meaninv.csv")
print(" - per_record_scores_meaninv.csv")
print(" - auc_ap_bootstrap_CI.txt")
print(" - config_meaninv.json")
print("\nMetrics table:")
print(df_metrics.round(3).to_string(index=False))


Saved to: /workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/CoughCrops/exports_loso_cropped_meaninv
 - roc_curve.png, pr_curve.png
 - cm_tau_star_maxF1.png, cm_tau_10%_FPR.png, cm_tau_20%_FPR.png
 - metrics_loso_cropped_meaninv.csv
 - per_record_scores_meaninv.csv
 - auc_ap_bootstrap_CI.txt
 - config_meaninv.json

Metrics table:
operating_point  threshold  ROC_AUC  PR_AUC    F1  precision_pos  recall_pos  specificity  TP  FN  FP  TN
 tau_star_maxF1  -2121.959    0.756   0.657 0.857          0.750       1.000        0.583  30   0  10  14
    tau_10%_FPR  -1326.423    0.756   0.657 0.000          0.000       0.000        0.875   0  30   3  21
    tau_20%_FPR  -1656.680    0.756   0.657 0.478          0.688       0.367        0.792  11  19   5  19


In [4]:
# CORAL alignment (unsupervised) for LOSO anomaly on cropped coughs
# - Align source TRAIN (COUGHVID+Coswara healthy crops) to TARGET (Kaggle healthy crops)
# - Fit Gaussian (Ledoit–Wolf) on CORAL-aligned source features
# - Score TARGET (Kaggle healthy & cancer) without labels
# - Aggregate per recording with MEAN, invert distances, report metrics

import numpy as np
from pathlib import Path
from collections import defaultdict
from PIL import Image
import torch, torchvision as tv
from torch.utils.data import Dataset, DataLoader
from sklearn.covariance import LedoitWolf
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    f1_score, confusion_matrix, roc_curve
)

# -------- paths --------
ROOT = Path("/workspaces/cmp9137-advanced-machine-learning/CMP9137 Advanced Machine Learning/CoughCrops/crops")
DIR_TR = ROOT / "train_healthy"
DIR_TH = ROOT / "test_healthy"
DIR_TC = ROOT / "test_cancer"
device = "cuda" if torch.cuda.is_available() else "cpu"

# -------- helpers --------
def list_imgs(d):
    exts = {".png",".jpg",".jpeg",".bmp",".tif",".tiff",".webp"}
    return [p for p in d.rglob("*") if p.suffix.lower() in exts]

def group_id_from_name(p: Path):
    s = p.stem
    return s.split("__x")[0] if "__x" in s else s

class ImgDataset(Dataset):
    def __init__(self, paths):
        self.paths = paths
        self.tf = tv.transforms.Compose([
            tv.transforms.Grayscale(num_output_channels=3),
            tv.transforms.ToTensor(),
            tv.transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        x = Image.open(self.paths[i]).convert("L")
        return self.tf(x), i

@torch.no_grad()
def embed(paths, backbone, batch=128, num_workers=0, device="cpu"):
    ds = ImgDataset(paths)
    dl = DataLoader(ds, batch_size=batch, shuffle=False, num_workers=num_workers)
    feats = np.zeros((len(paths), 512), dtype=np.float32)
    idx0 = 0
    for xb, ii in dl:
        xb = xb.to(device)
        f = backbone(xb).flatten(1).cpu().numpy().astype(np.float32)
        feats[idx0:idx0+f.shape[0]] = f
        idx0 += f.shape[0]
    return feats

def sym_eig_sqrt(mat, eps=1e-6, inv=False):
    # returns mat^{1/2} or mat^{-1/2}
    vals, vecs = np.linalg.eigh(mat)
    vals = np.clip(vals, eps, None)
    if inv:
        root = np.diag(1.0/np.sqrt(vals))
    else:
        root = np.diag(np.sqrt(vals))
    return vecs @ root @ vecs.T

def coral_align_source_to_target(Xs, Xt, eps=1e-6, with_mean=True):
    """
    Classic CORAL: Xs' = (Xs - ms) * Cs^{-1/2} * Ct^{1/2} + (mt if with_mean)
    """
    ms = Xs.mean(axis=0, keepdims=True)
    mt = Xt.mean(axis=0, keepdims=True)
    Xs_c = Xs - ms
    Xt_c = Xt - mt
    Cs = np.cov(Xs_c, rowvar=False) + eps*np.eye(Xs.shape[1], dtype=np.float32)
    Ct = np.cov(Xt_c, rowvar=False) + eps*np.eye(Xt.shape[1], dtype=np.float32)
    Cs_invhalf = sym_eig_sqrt(Cs, eps=eps, inv=True)
    Ct_half    = sym_eig_sqrt(Ct, eps=eps, inv=False)
    A = Cs_invhalf @ Ct_half
    Xs_aligned = Xs_c @ A
    if with_mean:
        Xs_aligned = Xs_aligned + mt
    return Xs_aligned, A, ms, mt

def fit_gaussian(X):
    mu = X.mean(axis=0, keepdims=True)
    Xc = X - mu
    lw = LedoitWolf().fit(Xc)
    return mu, lw.precision_

def mahalanobis_scores(X, mu, prec):
    Xc = X - mu
    return np.einsum("ij,jk,ik->i", Xc, prec, Xc)

def aggregate_mean(paths, scores):
    groups = defaultdict(list)
    for p,s in zip(paths, scores):
        groups[group_id_from_name(p)].append(float(s))
    gids, agg = [], []
    for gid, vals in groups.items():
        gids.append(gid); agg.append(float(np.mean(vals)))
    return np.array(gids), np.array(agg)

def pick_tau_maxF1(y, s):
    p, r, t = precision_recall_curve(y, s)
    f1 = 2*p*r/(p+r+1e-12)
    i = np.nanargmax(f1)
    return float(t[max(0, i-1)]) if i>0 else 0.0

def tau_at_fpr(y, s, target_fpr=0.10):
    neg = s[y==0]
    return float(np.quantile(neg, 1 - target_fpr)) if len(neg) else np.inf

def eval_at_tau(y, s, tau):
    yhat = (s >= tau).astype(int)
    cm = confusion_matrix(y, yhat, labels=[0,1])
    f1 = f1_score(y, yhat)
    return f1, cm

# -------- 1) list files --------
tr_paths = list_imgs(DIR_TR)
th_paths = list_imgs(DIR_TH)
tc_paths = list_imgs(DIR_TC)

# optional cap for training crops (speed)
MAX_TRAIN_CROPS = 12000
if len(tr_paths) > MAX_TRAIN_CROPS:
    from collections import defaultdict
    rec2 = defaultdict(list)
    for p in tr_paths: rec2[group_id_from_name(p)].append(p)
    k = max(3, int(MAX_TRAIN_CROPS / max(1,len(rec2))))
    rng = np.random.default_rng(123)
    sub = []
    for rec,lst in rec2.items():
        if len(lst) <= k: sub += lst
        else: sub += [lst[i] for i in rng.choice(len(lst), size=k, replace=False)]
    tr_paths_use = sub
else:
    tr_paths_use = tr_paths

# -------- 2) embed --------
resnet = tv.models.resnet18(weights=tv.models.ResNet18_Weights.IMAGENET1K_V1).to(device).eval()
backbone = torch.nn.Sequential(*(list(resnet.children())[:-1]))

X_tr = embed(tr_paths_use, backbone, device=device)
X_th = embed(th_paths,     backbone, device=device)  # target healthy (for CORAL stats & eval)
X_tc = embed(tc_paths,     backbone, device=device)  # target cancer  (eval)

# -------- 3) CORAL align source->target (using target healthy only) --------
Xs_corr, A, ms, mt = coral_align_source_to_target(X_tr, X_th, eps=1e-6, with_mean=True)

# -------- 4) Fit Gaussian on aligned source; score target (no transform) --------
mu, prec = fit_gaussian(Xs_corr)
s_th = mahalanobis_scores(X_th, mu, prec)
s_tc = mahalanobis_scores(X_tc, mu, prec)

# -------- 5) Aggregate per recording (mean) and invert distances --------
gid_th, rec_th = aggregate_mean(th_paths, s_th)
gid_tc, rec_tc = aggregate_mean(tc_paths, s_tc)
y_true  = np.concatenate([np.zeros_like(rec_th, dtype=int), np.ones_like(rec_tc, dtype=int)])
y_score = -np.concatenate([rec_th, rec_tc])  # invert: higher = more likely cancer

# -------- 6) Metrics --------
roc = roc_auc_score(y_true, y_score)
pr  = average_precision_score(y_true, y_score)
tau_star = pick_tau_maxF1(y_true, y_score)
thr10 = tau_at_fpr(y_true, y_score, target_fpr=0.10)
thr20 = tau_at_fpr(y_true, y_score, target_fpr=0.20)
f1_star, cm_star   = eval_at_tau(y_true, y_score, tau_star)
f1_10,   cm_10     = eval_at_tau(y_true, y_score, thr10)
f1_20,   cm_20     = eval_at_tau(y_true, y_score, thr20)

print("\n=== CORAL (source→Kaggle healthy), mean aggregation + inverted scores ===")
print(f"ROC-AUC={roc:.3f} | PR-AUC={pr:.3f}")
print(f"τ* (max-F1): tau={tau_star:.3f} | F1={f1_star:.3f} | CM (rows=healthy,cancer):\n{cm_star}")
print(f"10% FPR:     tau={thr10:.3f} | F1={f1_10:.3f} | CM:\n{cm_10}")
print(f"20% FPR:     tau={thr20:.3f} | F1={f1_20:.3f} | CM:\n{cm_20}")



=== CORAL (source→Kaggle healthy), mean aggregation + inverted scores ===
ROC-AUC=0.000 | PR-AUC=0.361
τ* (max-F1): tau=0.000 | F1=0.000 | CM (rows=healthy,cancer):
[[24  0]
 [30  0]]
10% FPR:     tau=-45.030 | F1=0.000 | CM:
[[20  4]
 [30  0]]
20% FPR:     tau=-45.049 | F1=0.000 | CM:
[[18  6]
 [30  0]]
